# False Positive Variants in Liquid Biopsy

## Exploring sequencing artifacts in cfDNA

**Team student research project · Repository maintainer: Victoria Makaryan**

This notebook explores sequencing evidence, read-position artifacts, dimensionality reduction, clustering, and candidate variant-review rules. The original clinical CSV is not distributed. The public edition clears embedded sample-level outputs and preserves the active research workflow with documented compatibility and row-identity fixes.

**Research prototype:** cluster-derived `GOOD` and `BAD` categories are interpretations, not independent clinical labels. Sensitivity, precision, and review-time savings have not been established by the supplied notebook.

### Navigation

1. Data loading and missingness
2. Cleaning and exploratory analysis
3. Preprocessing, PCA, and embeddings
4. Cluster comparison and historical interpretation
5. Original-scale comparisons and candidate triage rules

Read [methods and findings](../docs/methods-and-findings.md) and the [data guide](../docs/data.md) before running. Full team credits are linked in the [project presentation](https://docs.google.com/presentation/d/1VRVlxFJwOnT9711itlc5amiQRpV-2sJMkFSCQYh3cF0/edit).


## 1. Imports and private data

In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.stats import mannwhitneyu

In [ ]:
import os
from pathlib import Path

checkout = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
input_path = Path(os.environ.get("LIQUID_BIOPSY_DATA", str(checkout / "data/private/All_groups_in_one.csv")))
if not input_path.is_file():
    raise FileNotFoundError(
        "Private CSV not found. Set LIQUID_BIOPSY_DATA to an authorized local file; "
        "see docs/data.md. The public synthetic CSV is for the triage CLI only."
    )
df = pd.read_csv(input_path)


In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False)

# Data Cleaning

First, let's examine the data types and non-null values to understand the structure and identify potential issues.

In [ ]:
missing_percentages = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_percentages)

# Identify columns with a very high percentage of missing values (e.g., > 50%)
columns_to_consider_dropping = missing_percentages[missing_percentages > 50].index.tolist()
print(f"\nColumns with more than 50% missing values: {columns_to_consider_dropping}")

# Identify columns with a moderate percentage of missing values (e.g., between 5% and 50%)
columns_for_imputation = missing_percentages[(missing_percentages > 0) & (missing_percentages <= 50)].index.tolist()
print(f"Columns for potential imputation: {columns_for_imputation}")

### Missingness strategy

The source analysis dropped `gnomADg_AF_popmax` and `Patient_ID`, and used median imputation for selected numerical metrics. The public edition does not impute biological annotation strings. Identifiers are not modeling features. Inspect missingness in any new input rather than assuming the historical percentages apply.

In [ ]:
df_advanced_cleaned = df.copy()

# 1. Drop 'gnomADg_AF_popmax' due to high missing percentage
if 'gnomADg_AF_popmax' in df_advanced_cleaned.columns:
    df_advanced_cleaned = df_advanced_cleaned.drop(columns=['gnomADg_AF_popmax'])
    print("Dropped 'gnomADg_AF_popmax' column.")

# 2. Handle 'Patient_ID'
# Keep patient metadata outside the numerical modeling table.
# Patient identifiers are excluded from the modeling table.
if 'Patient_ID' in df_advanced_cleaned.columns:
    df_advanced_cleaned = df_advanced_cleaned.drop(columns=['Patient_ID'])
    print("Dropped 'Patient_ID' column due to high missing values.")

# 3. Impute remaining numerical columns with the median (robust to outliers)
# Identify numerical columns that still have NaNs
num_cols_to_impute = [
    col for col in ['Avg_Min_Distance_REF_reads', 'Avg_Min_Distance_ALT_reads', 'Strand_Bias']
    if col in df_advanced_cleaned.columns and df_advanced_cleaned[col].isnull().any()
]
for col in num_cols_to_impute:
    median_val = df_advanced_cleaned[col].median()
    df_advanced_cleaned[col] = df_advanced_cleaned[col].fillna(median_val)
    print(f"Imputed numerical column '{col}' with median: {median_val}")

# Leave annotation strings unchanged: they are not numerical modeling features.
print(f"Shape after numerical cleaning: {df_advanced_cleaned.shape}")
print((df_advanced_cleaned.isna().mean() * 100).sort_values(ascending=False).head())


### Duplicate working rows

Inspect and remove duplicate working rows before numerical exploration.

In [ ]:
# Check for duplicate rows in the advanced cleaned DataFrame
duplicate_rows_advanced = df_advanced_cleaned.duplicated().sum()
print(f"Number of duplicate rows found in df_advanced_cleaned: {duplicate_rows_advanced}")

# Drop duplicate rows if any exist
if duplicate_rows_advanced > 0:
    df_advanced_cleaned.drop_duplicates(inplace=True)
    print(f"df_advanced_cleaned shape after dropping duplicates: {df_advanced_cleaned.shape}")

# Display the first few rows of the advanced cleaned DataFrame
print("\nFirst 5 rows of the advanced cleaned DataFrame:")
display(df_advanced_cleaned.head())

## 2. Exploratory data analysis

### 1. Distribution Analysis

We'll analyze the distribution of numerical features using histograms and KDE plots, and categorical features using count plots. First, let's separate numerical and categorical columns.

In [ ]:
# Identify numerical and categorical columns
numerical_cols = df_advanced_cleaned.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_advanced_cleaned.select_dtypes(include='object').columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

# Exclude 'Unnamed: 0' if it's just an index from numerical columns for distribution plots
if 'Unnamed: 0' in numerical_cols:
    numerical_cols.remove('Unnamed: 0')


#### Numerical Features Distribution

In [ ]:
# Plot distributions for numerical features
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 4, i + 1) # Adjust subplot grid as needed
    sns.histplot(df_advanced_cleaned[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.show()

### 2. Outlier Detection

We'll use box plots to visually identify potential outliers in the numerical features. Outliers can skew analysis and model training, so understanding their presence is important.

In [ ]:
# Plot box plots for numerical features to detect outliers
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 4, i + 1) # Adjust subplot grid as needed
    sns.boxplot(y=df_advanced_cleaned[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel('')
plt.tight_layout()
plt.show()

### 3. Feature Correlation Analysis

Understanding the relationships between numerical features is crucial. We'll calculate and visualize the correlation matrix using a heatmap to identify highly correlated features.

In [ ]:
# Calculate the correlation matrix for numerical features
correlation_matrix = df_advanced_cleaned[numerical_cols].corr()

# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Feature Correlation Matrix')
plt.show()

### Raw feature variance

Variance describes spread in each feature's units. It is not a measure of predictive importance.

In [ ]:
# Calculate variance for numerical features
variance_analysis = df_advanced_cleaned[numerical_cols].var().sort_values(ascending=False)

print("Variance of Numerical Features (sorted descending):")
print(variance_analysis)

# Visualize variance
plt.figure(figsize=(10, 6))
sns.barplot(x=variance_analysis.values, y=variance_analysis.index, palette='viridis')
plt.title('Variance of Numerical Features')
plt.xlabel('Variance')
plt.ylabel('Feature')
plt.show()

## 3. Numerical preprocessing

Standardize numerical features. Robust scaling is an alternative for future comparison, not an additional fitted model here.

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the numerical columns
df_scaled = df_advanced_cleaned.copy()
df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])

print("First 5 rows of the DataFrame after StandardScaler:")
display(df_scaled[numerical_cols].head())

### Submitted log-transformation step

The source applies clipped `log1p` **after** standardization. Values below −1 are capped at −0.9999, potentially collapsing distinct observations. This order is preserved for transparency, not endorsed as an optimal transform. See the methods document for follow-up validation.

In [ ]:
import numpy as np

# Create a copy of the scaled DataFrame for log transformation
df_log_transformed = df_scaled.copy()

# Identify numerical columns suitable for log transformation (excluding ratios like VAF and potentially centered values like Strand_Bias)
log_transform_cols = [
    't_depth', 't_alt_count', 'n_depth', 'n_alt_count', 'variant_qual',
    'Avg_Min_Distance_ALT_reads', 'Avg_Min_Distance_REF_reads', 'ALT_≤3_Distance_Percent'
]

for col in log_transform_cols:
    if col in df_log_transformed.columns:
        # Cap values to be > -1 before applying log1p to avoid NaNs
        df_log_transformed[col] = np.log1p(np.maximum(df_log_transformed[col], -0.9999))
        print(f"Applied log1p transformation to '{col}'.")

print("\nFirst 5 rows of the DataFrame after Log Transformation (selected columns):")
display(df_log_transformed[log_transform_cols].head())

## 7. Outlier Handling

While scaling helps, explicit outlier handling can further improve model performance. We will use a simple Interquartile Range (IQR) based capping method to limit extreme values without completely removing data points, which is often a robust approach.

In [ ]:
# Create a copy of the log-transformed DataFrame for outlier handling
df_outliers_handled = df_log_transformed.copy()

# Define numerical columns for outlier capping
outlier_cols = numerical_cols # Use the same numerical columns identified earlier

for col in outlier_cols:
    if col in df_outliers_handled.columns:
        Q1 = df_outliers_handled[col].quantile(0.25)
        Q3 = df_outliers_handled[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Cap outliers
        df_outliers_handled[col] = np.where(df_outliers_handled[col] < lower_bound, lower_bound, df_outliers_handled[col])
        df_outliers_handled[col] = np.where(df_outliers_handled[col] > upper_bound, upper_bound, df_outliers_handled[col])
        print(f"Capped outliers for column '{col}'.")

print("\nFirst 5 rows of the DataFrame after Outlier Handling (selected columns):")
display(df_outliers_handled[outlier_cols].head())

## 4. Embeddings

Explore a two-dimensional UMAP projection of the processed numerical features. Predefined groups are used for visualization, not supplied as model features.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prepare the data for dimensionality reduction and clustering
# We will use the numerical columns from the df_outliers_handled DataFrame
data_for_dr_clustering = df_outliers_handled[numerical_cols].copy()

# Ensure no NaNs are present before UMAP/t-SNE
# This step is crucial if any previous transformation unexpectedly introduced NaNs
if data_for_dr_clustering.isnull().sum().sum() > 0:
    print("NaNs found in data_for_dr_clustering. Imputing with median...")
    for col in data_for_dr_clustering.columns:
        if data_for_dr_clustering[col].isnull().any():
            median_val = data_for_dr_clustering[col].median()
            data_for_dr_clustering[col] = data_for_dr_clustering[col].fillna(median_val)

# --- UMAP ---
print("Performing UMAP dimensionality reduction...")
reducer = umap.UMAP(n_components=2, random_state=42)
umap_embedding = reducer.fit_transform(data_for_dr_clustering)
df_outliers_handled['umap_x'] = umap_embedding[:, 0]
df_outliers_handled['umap_y'] = umap_embedding[:, 1]

# Visualize UMAP embedding
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='umap_x', y='umap_y',
    hue='group', # Use the 'group' column for coloring, if it makes sense for visualization
    data=df_outliers_handled,
    palette='tab10',
    s=10,
    alpha=0.7
)
plt.title('UMAP Projection of Numerical Features')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.legend(title='Group')
plt.show()

### PCA

Retain enough principal components to explain at least 95% of processed variance. Explained variance does not measure classification accuracy.

In [ ]:
# Apply PCA for dimensionality reduction and feature engineering
# We will choose a number of components that explain a significant portion of the variance (e.g., 95%)

pca = PCA(n_components=0.95, random_state=42) # Keep components that explain 95% of variance
pca_components = pca.fit_transform(data_for_dr_clustering)

print(f"Number of principal components explaining 95% variance: {pca_components.shape[1]}")
print("Explained variance ratio per component:")
print(pca.explained_variance_ratio_)

# Create a DataFrame for the PCA components
pca_df = pd.DataFrame(pca_components, columns=[f'PC_{i+1}' for i in range(pca_components.shape[1])])

# Add PCA components to our main DataFrame (if needed for later analysis, though clustering will use pca_components directly)
# df_outliers_handled = pd.concat([df_outliers_handled.reset_index(drop=True), pca_df], axis=1)

print("First 5 rows of PCA components:")
display(pca_df.head())

### PCA-based UMAP and t-SNE

These embeddings support visual exploration. Separation in two dimensions does not establish clinical discrimination.

In [ ]:
# --- UMAP with PCA components ---
print("Performing UMAP dimensionality reduction on PCA components...")
reducer_pca = umap.UMAP(n_components=2, random_state=42)
umap_embedding_pca = reducer_pca.fit_transform(pca_components)
df_outliers_handled['umap_pca_x'] = umap_embedding_pca[:, 0]
df_outliers_handled['umap_pca_y'] = umap_embedding_pca[:, 1]

# Visualize UMAP embedding with PCA
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='umap_pca_x', y='umap_pca_y',
    hue='group', # Still using 'group' for visualization
    data=df_outliers_handled,
    palette='tab10',
    s=10,
    alpha=0.7
)
plt.title('UMAP Projection with PCA Components')
plt.xlabel('UMAP Component 1 (from PCA)')
plt.ylabel('UMAP Component 2 (from PCA)')
plt.legend(title='Group')
plt.show()

# --- t-SNE with PCA components ---
print("\nPerforming t-SNE dimensionality reduction on PCA components...")
tsne_pca = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300)
tsne_embedding_pca = tsne_pca.fit_transform(pca_components)
df_outliers_handled['tsne_pca_x'] = tsne_embedding_pca[:, 0]
df_outliers_handled['tsne_pca_y'] = tsne_embedding_pca[:, 1]

# Visualize t-SNE embedding with PCA
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='tsne_pca_x', y='tsne_pca_y',
    hue='group', # Still using 'group' for visualization
    data=df_outliers_handled,
    palette='tab10',
    s=10,
    alpha=0.7
)
plt.title('t-SNE Projection with PCA Components')
plt.xlabel('t-SNE Component 1 (from PCA)')
plt.ylabel('t-SNE Component 2 (from PCA)')
plt.legend(title='Group')
plt.show()

## 5. Compare 4–6 clusters

Compare K-Means partitions on the PCA-based embeddings. Visual comparison is exploratory; this notebook does not establish a validated optimal k.

In [ ]:
K_values = range(4, 7) # Compare 4, 5, and 6 clusters

for k in K_values:
    print(f"\n--- Processing with K = {k} clusters ---")

    # K-Means on UMAP-PCA embedding
    kmeans_umap = KMeans(n_clusters=k, random_state=42, n_init=10)
    df_outliers_handled[f'kmeans_umap_k{k}'] = kmeans_umap.fit_predict(umap_embedding_pca)

    # K-Means on t-SNE-PCA embedding
    kmeans_tsne = KMeans(n_clusters=k, random_state=42, n_init=10)
    df_outliers_handled[f'kmeans_tsne_k{k}'] = kmeans_tsne.fit_predict(tsne_embedding_pca)

    # --- UMAP Plots ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f'UMAP Projections (PCA Features) for K={k} Clusters', fontsize=16)

    # Plot 1: UMAP colored by K-Means Clusters
    sns.scatterplot(
        x='umap_pca_x', y='umap_pca_y',
        hue=f'kmeans_umap_k{k}',
        data=df_outliers_handled,
        palette='tab10',
        s=50,
        alpha=0.7,
        ax=axes[0]
    )
    axes[0].set_title(f'K-Means Clusters (K={k})')
    axes[0].set_xlabel('UMAP Component 1 (from PCA)')
    axes[0].set_ylabel('UMAP Component 2 (from PCA)')
    axes[0].legend(title='Cluster')

    # Plot 2: UMAP colored by Original Group
    sns.scatterplot(
        x='umap_pca_x', y='umap_pca_y',
        hue='group',
        data=df_outliers_handled,
        palette='tab10',
        s=50,
        alpha=0.7,
        ax=axes[1]
    )
    axes[1].set_title('Original Group Categories')
    axes[1].set_xlabel('UMAP Component 1 (from PCA)')
    axes[1].set_ylabel('UMAP Component 2 (from PCA)')
    axes[1].legend(title='Group')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


### Historical five-cluster interpretation: explicit opt-in required

The source assigned clusters 0, 1, and 3 to `GOOD` and 2 and 4 to `BAD`, then excluded `WES_NO_cfDNA_NO`. Cluster IDs can change between fits. Inspect the current cluster composition against original evidence groups before reusing this mapping. The next cell deliberately stops here unless `REUSE_HISTORICAL_CLUSTER_LABELS` is set to `True` in this session. That opt-in only allows exploration; it does not validate the labels.

In [ ]:
REUSE_HISTORICAL_CLUSTER_LABELS = False


In [ ]:
if not REUSE_HISTORICAL_CLUSTER_LABELS:
    raise RuntimeError(
        "Historical cluster IDs are not portable. Inspect cluster composition, then "
        "explicitly opt in by setting REUSE_HISTORICAL_CLUSTER_LABELS = True."
    )

print("\n--- Creating 'GOOD'/'BAD' classification for K=5 clusters ---")

# Re-run K-Means with K=5 on UMAP-PCA embedding to ensure its availability
k_for_classification = 5
kmeans_umap_for_classification = KMeans(n_clusters=k_for_classification, random_state=42, n_init=10)
df_outliers_handled[f'kmeans_umap_k{k_for_classification}'] = kmeans_umap_for_classification.fit_predict(umap_embedding_pca)

# Filter out the specified group 'WES_NO_cfDNA_NO'
df_filtered = df_outliers_handled[df_outliers_handled['group'] != 'WES_NO_cfDNA_NO'].copy()

# Historical interpretation only; not independent clinical ground truth.
good_clusters = [0, 1, 3]
bad_clusters = [2, 4]

df_filtered['predicted_category'] = df_filtered[f'kmeans_umap_k{k_for_classification}'].apply(lambda x: 'GOOD' if x in good_clusters else ('BAD' if x in bad_clusters else 'UNKNOWN'))

print(f"Distribution of 'predicted_category' after filtering 'WES_NO_cfDNA_NO':")
display(df_filtered['predicted_category'].value_counts())

In [ ]:
# Preserve source row identity. Annotation-based merging can multiply rows.
assert df.index.is_unique and df_filtered.index.is_unique
assert df_filtered.index.isin(df.index).all()
df_original_with_predictions_filtered = df.join(
    df_filtered[['predicted_category']], how='inner', validate='one_to_one'
)
assert len(df_original_with_predictions_filtered) == len(df_filtered)
print(f"Retained {len(df_original_with_predictions_filtered)} original rows without join expansion.")


## 6. Original-scale feature comparisons

Compare original-scale metrics across the historical cluster interpretations. Mann–Whitney U tests are exploratory; these are not independently labeled TP/FP comparisons.

#### Scatterplots on Original Data, Categorized by Predicted Category

In [ ]:
print("Descriptive Statistics for 't_alt_count' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['t_alt_count'].describe())

# Boxplot for t_alt_count
plt.figure(figsize=(8, 6))
sns.boxplot(x='predicted_category', y='t_alt_count', data=df_original_with_predictions_filtered, palette={'GOOD': 'lightgreen', 'BAD': 'salmon'})
sns.stripplot(x='predicted_category', y='t_alt_count', data=df_original_with_predictions_filtered, color='black', size=3, jitter=0.2, alpha=0.5)
plt.title('Original Data: Distribution of t_alt_count by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('t_alt_count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0,300)
plt.show()

# Mann-Whitney U test for t_alt_count
good_values_t_alt_count = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'GOOD']['t_alt_count']
bad_values_t_alt_count = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'BAD']['t_alt_count']

if len(good_values_t_alt_count) > 0 and len(bad_values_t_alt_count) > 0:
    statistic, p_value = mannwhitneyu(good_values_t_alt_count, bad_values_t_alt_count, alternative='two-sided')
    print(f"  Mann-Whitney U test for t_alt_count: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")
    if p_value < 0.05:
        print(f"  -> Statistically significant difference (p < 0.05).")
    else:
        print(f"  -> No statistically significant difference (p >= 0.05).")
else:
    print(f"  Cannot perform Mann-Whitney U test: one or both categories are empty for t_alt_count.")

In [ ]:
print("Descriptive Statistics for 'variant_qual' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['variant_qual'].describe())

# Boxplot for variant_qual
plt.figure(figsize=(8, 6))
sns.boxplot(x='predicted_category', y='variant_qual', data=df_original_with_predictions_filtered, palette={'GOOD': 'lightgreen', 'BAD': 'salmon'})
sns.stripplot(x='predicted_category', y='variant_qual', data=df_original_with_predictions_filtered, color='black', size=3, jitter=0.2, alpha=0.5)
plt.title('Original Data: Distribution of variant_qual by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('variant_qual')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0,2000)
plt.show()

# Mann-Whitney U test for variant_qual
good_values_variant_qual = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'GOOD']['variant_qual']
bad_values_variant_qual = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'BAD']['variant_qual']

if len(good_values_variant_qual) > 0 and len(bad_values_variant_qual) > 0:
    statistic, p_value = mannwhitneyu(good_values_variant_qual, bad_values_variant_qual, alternative='two-sided')
    print(f"  Mann-Whitney U test for variant_qual: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")
    if p_value < 0.05:
        print(f"  -> Statistically significant difference (p < 0.05).")
    else:
        print(f"  -> No statistically significant difference (p >= 0.05).")
else:
    print(f"  Cannot perform Mann-Whitney U test: one or both categories are empty for variant_qual.")

In [ ]:
print("Descriptive Statistics for 't_depth' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['t_depth'].describe())

# Boxplot for t_depth
plt.figure(figsize=(8, 6))
sns.boxplot(x='predicted_category', y='t_depth', data=df_original_with_predictions_filtered, palette={'GOOD': 'lightgreen', 'BAD': 'salmon'})
sns.stripplot(x='predicted_category', y='t_depth', data=df_original_with_predictions_filtered, color='black', size=3, jitter=0.2, alpha=0.5)
plt.title('Original Data: Distribution of t_depth by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('t_depth')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0,10000)
plt.show()

# Mann-Whitney U test for t_depth
good_values_t_depth = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'GOOD']['t_depth']
bad_values_t_depth = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'BAD']['t_depth']

if len(good_values_t_depth) > 0 and len(bad_values_t_depth) > 0:
    statistic, p_value = mannwhitneyu(good_values_t_depth, bad_values_t_depth, alternative='two-sided')
    print(f"  Mann-Whitney U test for t_depth: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")
    if p_value < 0.05:
        print(f"  -> Statistically significant difference (p < 0.05).")
    else:
        print(f"  -> No statistically significant difference (p >= 0.05).")
else:
    print(f"  Cannot perform Mann-Whitney U test: one or both categories are empty for t_depth.")

In [ ]:
print("Descriptive Statistics for 'ALT_≤3_Distance_Percent' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['ALT_≤3_Distance_Percent'].describe())

# Boxplot for ALT_≤3_Distance_Percent
plt.figure(figsize=(8, 6))
sns.boxplot(x='predicted_category', y='ALT_≤3_Distance_Percent', data=df_original_with_predictions_filtered, palette={'GOOD': 'lightgreen', 'BAD': 'salmon'})
sns.stripplot(x='predicted_category', y='ALT_≤3_Distance_Percent', data=df_original_with_predictions_filtered, color='black', size=3, jitter=0.2, alpha=0.5)
plt.title('Original Data: Distribution of ALT_≤3_Distance_Percent by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('ALT_≤3_Distance_Percent')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Mann-Whitney U test for ALT_≤3_Distance_Percent
good_values_alt_dist = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'GOOD']['ALT_≤3_Distance_Percent']
bad_values_alt_dist = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'BAD']['ALT_≤3_Distance_Percent']

if len(good_values_alt_dist) > 0 and len(bad_values_alt_dist) > 0:
    statistic, p_value = mannwhitneyu(good_values_alt_dist, bad_values_alt_dist, alternative='two-sided')
    print(f"  Mann-Whitney U test for ALT_≤3_Distance_Percent: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")
    if p_value < 0.05:
        print(f"  -> Statistically significant difference (p < 0.05).")
    else:
        print(f"  -> No statistically significant difference (p >= 0.05).")
else:
    print(f"  Cannot perform Mann-Whitney U test: one or both categories are empty for ALT_≤3_Distance_Percent.")

In [ ]:
print("Descriptive Statistics for 'Avg_Min_Distance_ALT_reads' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['Avg_Min_Distance_ALT_reads'].describe())

# Boxplot for Avg_Min_Distance_ALT_reads
plt.figure(figsize=(8, 6))
sns.boxplot(x='predicted_category', y='Avg_Min_Distance_ALT_reads', data=df_original_with_predictions_filtered, palette={'GOOD': 'lightgreen', 'BAD': 'salmon'})
sns.stripplot(x='predicted_category', y='Avg_Min_Distance_ALT_reads', data=df_original_with_predictions_filtered, color='black', size=3, jitter=0.2, alpha=0.5)
plt.title('Original Data: Distribution of Avg_Min_Distance_ALT_reads by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('Avg_Min_Distance_ALT_reads')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Mann-Whitney U test for Avg_Min_Distance_ALT_reads
good_values_avg_min_dist = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'GOOD']['Avg_Min_Distance_ALT_reads']
bad_values_avg_min_dist = df_original_with_predictions_filtered[df_original_with_predictions_filtered['predicted_category'] == 'BAD']['Avg_Min_Distance_ALT_reads']

if len(good_values_avg_min_dist) > 0 and len(bad_values_avg_min_dist) > 0:
    statistic, p_value = mannwhitneyu(good_values_avg_min_dist, bad_values_avg_min_dist, alternative='two-sided')
    print(f"  Mann-Whitney U test for Avg_Min_Distance_ALT_reads: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")
    if p_value < 0.05:
        print(f"  -> Statistically significant difference (p < 0.05).")
    else:
        print(f"  -> No statistically significant difference (p >= 0.05).")
else:
    print(f"  Cannot perform Mann-Whitney U test: one or both categories are empty for Avg_Min_Distance_ALT_reads.")

# Scatterplot on original data

In [ ]:
print("\n--- Scatterplots for specified metric pairs (GOOD vs. BAD) ---")

# Scatterplot 1: t_alt_count vs variant_qual
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t_alt_count',
    y='variant_qual',
    hue='predicted_category',
    data=df_original_with_predictions_filtered,
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50, # Marker size
    alpha=0.7 # Transparency
)
plt.title('t_alt_count vs variant_qual by Predicted Category')
plt.xlabel('t_alt_count')
plt.ylabel('variant_qual')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category')
plt.ylim(0, 2000)
plt.xlim(0, 100)
plt.show()

# Scatterplot 2: ALT_≤3_Distance_Percent vs Avg_Min_Distance_ALT_reads
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='ALT_≤3_Distance_Percent',
    y='Avg_Min_Distance_ALT_reads',
    hue='predicted_category',
    data=df_original_with_predictions_filtered,
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50, # Marker size
    alpha=0.7 # Transparency
)
plt.title('ALT_≤3_Distance_Percent vs Avg_Min_Distance_ALT_reads by Predicted Category')
plt.xlabel('ALT_≤3_Distance_Percent')
plt.ylabel('Avg_Min_Distance_ALT_reads')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category')
plt.show()

In [ ]:
# Re-creating the scatterplot for t_alt_count vs variant_qual
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t_alt_count',
    y='variant_qual',
    hue='predicted_category',
    data=df_original_with_predictions_filtered, # Use the full DataFrame
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50, # Marker size
    alpha=0.7 # Transparency
)

# Identify the specific points to highlight: 'BAD' points where Avg_Min_Distance_ALT_reads < 20
highlight_condition = (
    (df_original_with_predictions_filtered['predicted_category'] == 'BAD') &
    (df_original_with_predictions_filtered['Avg_Min_Distance_ALT_reads'] < 20)
)
highlight_points = df_original_with_predictions_filtered[highlight_condition].copy()

# Overlay these highlight points with blue crosses
sns.scatterplot(
    x='t_alt_count',
    y='variant_qual',
    data=highlight_points,
    color='blue',
    marker='o', # Highlighted subset
    s=100, # Larger size for emphasis
    alpha=1.0, # Full opacity for emphasis
    label='BAD points with Avg_Min_Distance_ALT_reads < 20'
)

plt.title('t_alt_count vs variant_qual by Predicted Category (with highlighted BAD points)', fontsize=14)
plt.xlabel('t_alt_count')
plt.ylabel('variant_qual')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Category', loc='upper right')
plt.xlim(0,100)
plt.ylim(0,2000)
plt.show()

### Visual inspection after a selected artifact-like subset is omitted

This plot intentionally excludes some `BAD` points based on read-edge distance. It must not be interpreted as an unbiased performance evaluation.

In [ ]:
print("\n--- Scatterplots for specified metric pairs (Original Data, GOOD vs. BAD) ---")

# Define the condition for removing points from the main scatterplot
removal_condition = (
    (df_original_with_predictions_filtered['predicted_category'] == 'BAD') &
    (df_original_with_predictions_filtered['Avg_Min_Distance_ALT_reads'] < 20)
)

# Create a filtered DataFrame for the first scatterplot
df_for_first_scatterplot = df_original_with_predictions_filtered[~removal_condition].copy()

# Scatterplot 1: t_alt_count vs variant_qual
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t_alt_count',
    y='variant_qual',
    hue='predicted_category',
    data=df_for_first_scatterplot, # Use the filtered DataFrame here
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50, # Marker size
    alpha=0.7 # Transparency
)

# Identify points for highlighting (this condition remains as per the last state)
points_to_highlight_unfiltered = df_original_with_predictions_filtered[
    (df_original_with_predictions_filtered['ALT_≤3_Distance_Percent'] >= 20) &
    (df_original_with_predictions_filtered['Avg_Min_Distance_ALT_reads'] < 20)
]

# Filter the points to highlight, ensuring they are not among those removed from the main plot
points_to_highlight_final = points_to_highlight_unfiltered[~removal_condition].copy()

# Overlay these (filtered) highlight points on the first scatterplot with a blue color and different marker
sns.scatterplot(
    x='t_alt_count',
    y='variant_qual',
    data=points_to_highlight_final, # Use the filtered highlight DataFrame
    color='blue',
    marker='X', # Use a different marker to distinguish
    s=100, # Larger size for emphasis
    alpha=0.9, # Higher transparency for emphasis
    label='ALT_≤3_Distance_Percent >= 20 & Avg_Min_Distance_ALT_reads < 20 (Not Removed)'
)

plt.title('Original Data: t_alt_count vs variant_qual by Predicted Category (Filtered)')
plt.xlabel('t_alt_count')
plt.ylabel('variant_qual')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category / Highlighted')
plt.xlim(0,100)
plt.ylim(0,2000)
plt.show()

# Scatterplot 2: ALT_≤3_Distance_Percent vs Avg_Min_Distance_ALT_reads
# This plot uses the original df_original_with_predictions_filtered as the request for removal was specific to the first plot's condition.
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='ALT_≤3_Distance_Percent',
    y='Avg_Min_Distance_ALT_reads',
    hue='predicted_category',
    data=df_original_with_predictions_filtered, # Uses the original unfiltered data for this plot
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50, # Marker size
    alpha=0.7 # Transparency
)
plt.title('Original Data: ALT_≤3_Distance_Percent vs Avg_Min_Distance_ALT_reads by Predicted Category')
plt.xlabel('ALT_≤3_Distance_Percent')
plt.ylabel('Avg_Min_Distance_ALT_reads')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category')
plt.show()

### Linear boundary fitted to interpreted cluster labels

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the feature pair for this plot
feature1_name = 'ALT_≤3_Distance_Percent'
feature2_name = 'Avg_Min_Distance_ALT_reads'

# Prepare data for plotting and classification
plot_df = df_original_with_predictions_filtered[[feature1_name, feature2_name, 'predicted_category']].dropna().copy()
X = plot_df[[feature1_name, feature2_name]]
y = plot_df['predicted_category'].map({'GOOD': 1, 'BAD': 0}) # Encode interpreted categories for visualization

# Scale features for classification
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train a Linear Classifier (Logistic Regression)
lr = LogisticRegression(random_state=42)
lr.fit(X_scaled, y)

# Get the coefficients and intercept from the scaled data
coef_scaled = lr.coef_[0]
intercept_scaled = lr.intercept_[0]

# Plotting
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=feature1_name,
    y=feature2_name,
    hue='predicted_category',
    data=plot_df,
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50,
    alpha=0.7
)

# To correctly plot the decision boundary on the original scale:
x_min_orig, x_max_orig = X[feature1_name].min() - (X[feature1_name].std() * 0.5), X[feature1_name].max() + (X[feature1_name].std() * 0.5)
y_min_orig, y_max_orig = X[feature2_name].min() - (X[feature2_name].std() * 0.5), X[feature2_name].max() + (X[feature2_name].std() * 0.5)

x_plot = np.linspace(x_min_orig, x_max_orig, 100)

# Convert x_plot to scaled
x_plot_scaled = (x_plot - scaler.mean_[0]) / scaler.scale_[0]

# Calculate y_scaled from the linear boundary equation
y_plot_scaled = (-coef_scaled[0] * x_plot_scaled - intercept_scaled) / coef_scaled[1]

# Convert y_plot_scaled back to original
y_plot = (y_plot_scaled * scaler.scale_[1]) + scaler.mean_[1]

plt.plot(x_plot, y_plot, 'b--', label='Linear Decision Boundary')

plt.title(f'Original Data: {feature1_name} vs {feature2_name} with Linear Boundary')
plt.xlabel(feature1_name)
plt.ylabel(feature2_name)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category / Boundary')
plt.show()

# Display the equation of the linear decision boundary
print(f"Equation of the Linear Decision Boundary (in terms of original features):")
print(f"  {coef_scaled[0]:.3f} * ((X - {scaler.mean_[0]:.3f}) / {scaler.scale_[0]:.3f}) + ")
print(f"  {coef_scaled[1]:.3f} * ((Y - {scaler.mean_[1]:.3f}) / {scaler.scale_[1]:.3f}) + {intercept_scaled:.3f} = 0")
print(f"Where X is '{feature1_name}' and Y is '{feature2_name}'.")

### Nonlinear boundary fitted to interpreted cluster labels

In [ ]:
# Non-linear Classifier (Support Vector Machine with RBF kernel)
svc_rbf = SVC(kernel='rbf', random_state=42, probability=True)
svc_rbf.fit(X_scaled, y)

# Create meshgrid for plotting decision boundaries
x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                       np.arange(y_min, y_max, 0.02))

# Make predictions for the meshgrid
Z_nonlinear = svc_rbf.predict(np.c_[xx.ravel(), yy.ravel()])
Z_nonlinear = Z_nonlinear.reshape(xx.shape)

# Plotting
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=feature1_name,
    y=feature2_name,
    hue='predicted_category',
    data=plot_df,
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    s=50,
    alpha=0.7
)

# Transform meshgrid coordinates back to original scale for plotting
xx_orig = xx * scaler.scale_[0] + scaler.mean_[0]
yy_orig = yy * scaler.scale_[1] + scaler.mean_[1]

plt.contourf(xx_orig, yy_orig, Z_nonlinear, alpha=0.3,
             cmap=plt.cm.RdYlGn, levels=[-0.5, 0.5, 1.5])
plt.contour(xx_orig, yy_orig, Z_nonlinear, levels=[0.5], colors='purple', linestyles='-', linewidths=2)

plt.title(f'Original Data: {feature1_name} vs {feature2_name} with Non-Linear Boundary')
plt.xlabel(feature1_name)
plt.ylabel(feature2_name)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Predicted Category')
plt.show()

print("Equation of the Non-Linear Decision Boundary (RBF SVM):")
print("  For an RBF kernel Support Vector Machine, the decision boundary is generally complex and cannot be represented by a simple algebraic equation of the form y=f(x).")
print("  Instead, it is defined implicitly by the kernel function and support vectors, creating non-linear regions as visualized above.")

## 7. Candidate review actions

Apply the submitted original-scale rules: quality >380 and alternate count >31 for `AUTO-PASS`; otherwise alternate-read edge distance <20 for `AUTO-REJECT`; otherwise `MANUAL_REVIEW`. Pass takes precedence. These exploratory thresholds are not clinically validated. The extracted Python module separately demonstrates this logic using invented data.

In [ ]:
# Initialize 'final_classification' column with 'MANUAL_REVIEW'
df_original_with_predictions_filtered['final_classification'] = 'MANUAL_REVIEW'

# 1. Apply AUTO-PASS conditions
# Criteria: variant_qual > 380 AND t_alt_count > 31
auto_pass_condition = (
    (df_original_with_predictions_filtered['variant_qual'] > 380) &
    (df_original_with_predictions_filtered['t_alt_count'] > 31)
)
df_original_with_predictions_filtered.loc[auto_pass_condition, 'final_classification'] = 'AUTO-PASS'

# 2. Apply AUTO-REJECT conditions
# Criteria: Avg_Min_Distance_ALT_reads < 20 (only for those not already AUTO-PASS)
auto_reject_condition = (
    (df_original_with_predictions_filtered['final_classification'] == 'MANUAL_REVIEW') &
    (df_original_with_predictions_filtered['Avg_Min_Distance_ALT_reads'] < 20)
)
df_original_with_predictions_filtered.loc[auto_reject_condition, 'final_classification'] = 'AUTO-REJECT'

# The remaining entries will retain 'MANUAL_REVIEW' as assigned initially.

print("--- Final Classification Summary ---")
display(df_original_with_predictions_filtered['final_classification'].value_counts())

print("\nFirst 5 rows of the DataFrame with the new 'final_classification' column:")
display(df_original_with_predictions_filtered[['Order_ID_cfDNA', 'HGVSc', 'HGVSp_Short', 'variant_qual', 't_alt_count', 'Avg_Min_Distance_ALT_reads', 'final_classification']].head())

In [ ]:
print("Descriptive Statistics for 't_alt_count' by Predicted Category (Original Data)")
display(df_original_with_predictions_filtered.groupby('predicted_category')['t_alt_count'].describe())

# Boxplot for t_alt_count
plt.figure(figsize=(8, 6))

sns.boxplot(
    x='predicted_category',
    y='t_alt_count',
    data=df_original_with_predictions_filtered,
    palette={'GOOD': 'lightgreen', 'BAD': 'salmon'},
    showfliers=False   # removes the big outlier circles
)

sns.stripplot(
    x='predicted_category',
    y='t_alt_count',
    data=df_original_with_predictions_filtered,
    color='black',
    size=3,
    jitter=0.2,
    alpha=0.5
)

plt.title('Original Data: Distribution of t_alt_count by Predicted Category (GOOD vs. BAD)')
plt.xlabel('Predicted Category')
plt.ylabel('t_alt_count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 300)
plt.show()

# Mann-Whitney U test for t_alt_count
good_values_t_alt_count = df_original_with_predictions_filtered[
    df_original_with_predictions_filtered['predicted_category'] == 'GOOD'
]['t_alt_count']

bad_values_t_alt_count = df_original_with_predictions_filtered[
    df_original_with_predictions_filtered['predicted_category'] == 'BAD'
]['t_alt_count']

if len(good_values_t_alt_count) > 0 and len(bad_values_t_alt_count) > 0:
    statistic, p_value = mannwhitneyu(
        good_values_t_alt_count,
        bad_values_t_alt_count,
        alternative='two-sided'
    )
    print(f"  Mann-Whitney U test for t_alt_count: Statistic = {statistic:.3f}, P-value = {p_value:.3e}")

    if p_value < 0.05:
        print("  -> Statistically significant difference (p < 0.05).")
    else:
        print("  -> No statistically significant difference (p >= 0.05).")
else:
    print("  Cannot perform Mann-Whitney U test: one or both categories are empty for t_alt_count.")

## Interpretation and next steps

The public edition preserves exploratory methods without asserting unverified clinical accuracy. Re-evaluate preprocessing and cluster stability, use independently adjudicated labels, and validate thresholds on held-out samples before quantifying sensitivity, precision, or manual-review savings. Historical aggregate outputs and the original row-count discrepancy are documented in [methods and findings](../docs/methods-and-findings.md).

Do not commit an executed notebook containing private row-level tables or identifiers. The checked-in notebook intentionally has no stored outputs.
